## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [1]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

# Detectăm ROOT-ul proiectului căutând fișierul .env în sus
def find_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / ".env").exists():
            return parent
    raise FileNotFoundError("Nu am găsit .env — verifică structura proiectului.")

PROJECT_ROOT = find_root(Path.cwd())
os.chdir(PROJECT_ROOT)

BUBBLES_DIR = PROJECT_ROOT / "data" / "bubbles"
VECTOR_DIR  = PROJECT_ROOT / "assets" / "vectorstores"
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

print("ROOT:", PROJECT_ROOT)

c:\Users\georg\OneDrive\Dokument\Claude\Projects\Cursul Inginerie Ai\echochamber-project-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT: c:\Users\georg\OneDrive\Dokument\Claude\Projects\Cursul Inginerie Ai\echochamber-project-team3


## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [2]:
MY_BUBBLE_FILE = "intelectual_critic.jsonl"   # student_06

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: intelectual_critic
Texte: 49


,id,agent,text
0,yt_olpOFshMJD0_UgznnPuVIJ6HcddfPbN4AaABAg,Intelectual-critic,Mă voi realizați că votul s-a încheiat și Nicu...
1,yt_cok3YTTn8sg_Ugw9joa6s_uddLOSdCZ4AaABAg,Intelectual-critic,"Catu sa plateasca, daca nu se pruc probe ale u..."
2,yt__xecHPdEhuI_UgxEpwxIKvuelKuQ0r94AaABAg,Intelectual-critic,Acum vă eu pe cei care ati votat altceva decat...
3,yt_iH8jB4NlV9Y_UgxUZokcnI7ENgQqHPZ4AaABAg,Intelectual-critic,18:23 poate sa imi explice și mie un Simionist...
4,yt_yEuctxNb4O0_UgxPNjxaWPqCBL5WUAB4AaABAg,Intelectual-critic,Bun! S-a desfășurat aceasta întâlnire. S-a lua...


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [3]:
texts    = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Mă voi realizați că votul s-a încheiat și Nicușor Dan este actualul președinte ales de majoritate prin vot fără incidente confirmat și de CCR?. Din partidul POT s-au retras mai toți membrii importanți căutând alte oportunități în partide PSD sau PNL sau USR ceea ce este firesc dacă le merge mintea de ce să nu ocupe un post bun spre beneficiul cetățenilor mai ales dacă au umbrela unor partide puternice?.


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [4]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.91it/s]

Număr texte: 49
Dimensiune embeddings: (49, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [5]:
# Bula mea (T6_intelectual_critic) are 48 de texte.
# Au fost generați 48 de vectori, câte unul per text.
# A doua valoare din embeddings.shape (384) reprezintă numărul de dimensiuni
# ale spațiului vectorial — fiecare text este codificat ca un vector de 384 de numere.
print("Texte în bulă:", len(texts))
print("Vectori generați:", embeddings.shape[0])
print("Dimensiuni per vector:", embeddings.shape[1])

Texte în bulă: 49
Vectori generați: 49
Dimensiuni per vector: 384


## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [6]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)

faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: c:\Users\georg\OneDrive\Dokument\Claude\Projects\Cursul Inginerie Ai\echochamber-project-team3\assets\vectorstores\intelectual_critic
Vectori în index: 49


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [7]:
faiss_path = out_dir / "index.faiss"
pkl_path   = out_dir / "index.pkl"

print("index.faiss există:", faiss_path.exists())
print("index.pkl există:  ", pkl_path.exists())
print("index.ntotal este egal cu numărul de texte:", index.ntotal == len(texts))
print(f"  → {index.ntotal} vectori == {len(texts)} texte")

index.faiss există: True
index.pkl există:   True
index.ntotal este egal cu numărul de texte: True
  → 49 vectori == 49 texte


## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [8]:
# Text nou introdus în aplicație
# Ales pentru a testa agentul T6_intelectual_critic:
# o afirmație care cere evaluare pe bază de dovezi, nu de emoție

input_text = "Politicienii fac promisiuni fără să prezinte un program concret sau date verificabile."

In [9]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [10]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.441
Text: Pt astia care zic justitie fara politica, trebuie sa fie ceck and balance in stat pentru a evita deraieri.

Rezultat 2
Scor: 0.407
Text: Nu a aratat o dovoda clara doar isi continua narativa. In video sunt doar niste oameni, habar nu am cine sunt (ca nu sunt prezentati), care isi dau cu parerea fara sa produca un act sau video cu ce spun. Iar ceea ce spun este de domeniul SF-ului. Nu mentineaza deloc ajutorul acordat de Romania Moldovei si programele dintre cele 2 tari. Apropo, daca Moldova intre in UE atunci nu o sa mai fie nevoie de granite, o sa fie un fel de Unire. Ce vad in partidul AUR doar oameni care fac orice pentru pute

Rezultat 3
Scor: 0.385
Text: Episodul 2 vine cu mai multe vorbe goale si minciuni decat primul. Se vede ca Simion si restul membrilor AUR din clip habar nu au cum functioneaza sistemul informatic de votare si ce restrictii exista special pentru a impiedica votul multiplu.

Rezultat 4
Scor: 0.367
Text: 1:01:06 Care e problema?Oric

### Observații retrieval T6_intelectual_critic

- **Relevanță**: textele recuperate ar trebui să ceară dovezi, să conteste afirmații fără suport sau să analizeze comportamentul actorilor politici.
- **Vocea agentului**: dacă rezultatele sunt analitice și detașate (nu emoționale/conspirative), indexul funcționează corect.
- **Text slab**: dacă un rezultat sună mai degrabă ca T1 (laudativ) sau T2 (furios), notează ID-ul și adaugă-l în `REMOVE_IDS` din C5_01.

In [ ]:
# Notițe după inspecție:
# Rezultate relevante din 5: 5
# Textele recuperate exprimă vocea agentului T6: da
# Texte slabe observate (ID): niciun text slab identificat